In [4]:
import math
import mesa
import pandas as pd
import matplotlib.pyplot as plt


In [5]:
class AgenteViajero(mesa.Agent):
    def __init__(self, model, tiene_carro, hora_salida):
        super().__init__(model)
        self.estado = "en_casa"
        self.tiene_carro = tiene_carro
        self.hora_salida = hora_salida
        self.duracion = 30 if tiene_carro else 60
        self.tiempo_viaje = None
        self.hora_llegada = None

    def llegar(self):
        self.estado = "llego"
        self.hora_llegada = self.model.minuto_actual
        self.tiempo_viaje = self.hora_llegada - self.hora_salida

    def step(self):
        if self.estado == "llego":
            return
        ahora = self.model.minuto_actual
        if self.estado == "en_casa" and ahora >= self.hora_salida:
            self.estado = "viajando"
        if self.estado == "viajando" and ahora >= self.hora_salida + self.duracion:
            self.llegar()


In [6]:
class Modelo(mesa.Model):
    def __init__(self, n=100, hora_inicio=6*60, minutos_por_step=1, seed=None):
        super().__init__(seed=seed)
        self.t = 0
        self.hora_inicio = hora_inicio
        self.dt = minutos_por_step
        for _ in range(n):
            AgenteViajero(
                self,
                tiene_carro=self.random.random() < 0.4,
                hora_salida=hora_inicio + self.random.randint(0, 90),
            )

    @property
    def minuto_actual(self):
        return self.hora_inicio + self.t * self.dt

    def step(self):
        self.agents.shuffle_do("step")
        self.t += 1


In [8]:
m = Modelo(n=100, seed=42)
for _ in range(180):          # 3 horas
    m.step()

df = pd.DataFrame([
    {"carro": a.tiene_carro, "salida": a.hora_salida,
     "llegada": a.hora_llegada, "tiempo_viaje": a.tiempo_viaje}
    for a in m.agents
])
df.groupby("carro")["tiempo_viaje"].mean()
df


,carro,salida,llegada,tiempo_viaje
0,False,363,423,60
1,False,391,451,60
2,True,373,403,30
3,False,429,489,60
4,True,414,444,30
...,...,...,...,...
95,False,360,420,60
96,True,440,470,30
97,True,368,398,30
98,False,402,462,60
